# Test `multicore_mesh_pipeline_final.py` on Windows

This notebook installs dependencies and runs the pipeline script with sample parameters using `python` rather than shell executables.

In [1]:
#Optionally install required libraries
!pip install numpy scikit-image trimesh tqdm
#In VASTlite (tested Version is 1.50) open the remote control API window and enable the API
#Before using the script, cd to the directory containing "vast_comm.py", "vast_param.py", "vast_parser.py", "vast_control.py", "multicore_mesh_pipeline_info_mip_edgepads_with_mip_scale----final_v02.py" and "glue_and_scale_mesh----final_v01.py"

   ---------------------------------------- 0.0/709.3 kB ? eta -:--:--
   --------------------------------------- 709.3/709.3 kB 14.4 MB/s eta 0:00:00


In [55]:
# Run the pipeline for segment 1 with 8 workers at mip level 0 with a chunk size of 512x512x32 (adjust if more workers or ram available)
# In case of OOM Error adjust either chunksize or num_workers
#
# Parameters are:
# "--host" and "--port"         (Host ip and port number to connect to VAST-API, default is '127.0.0.1' and '22081')
# "--segment", "1"              (Which segment to export, 0 is background)
# "--workers", "8"              (How many workers you use for parallel mesh generation)
# "--chunk", "512", "512", "32" (How big one chunk is in XYZ)
# "--overlap", "1"              (How much overlap between chunks, 1 Pixel should be sufficient)
# "--mip", "0"                  (Mip-level for mesh export, downscaled by half in XY for each mip +1)
# This will first download numpy arrays of each chunks seg_data and place them at ./tmp_dbg, next it will parallel process each array and save its mesh to ./tmp_dbg/meshes
#
# These meshes you can either glue together or load partially to your preferred 3D-Rendering software (eg Blender)
import sys
sys.executable, "multicore_mesh_pipeline_info_mip_edgepads_with_mip_scale----final_v02.py", "--segment", "1", "--workers", "8", "--chunk", "512", "512", "32", "--overlap", "1", "--mip", "0"
!python multicore_mesh_pipeline_info_mip_edgepads_with_mip_scale----final_v02.py --segment 1 --workers 8 --chunk 512 512 32 --overlap 1 --mip 0

=== VOLUME INFO ===
Full-res dims: x=9444, y=10512, z=295
MIP 0:       x=9444, y=10512, z=295
Voxel size:     x=50.0, y=50.0, z=200.0
=== SEGMENT BBOX ===
Full-res: min=(492,328,0), max=(9248,9888,294)
MIP 0:     min=(492,328,0), max=(9248,9888,294)
Total chunks: 3420
=== DONE ===
Masks in: tmp_dbg_12
Meshes in: tmp_dbg_12\meshes



Fetching: 100%|##########| 3420/3420 [1:05:41<00:00,  1.15s/it]

Meshing: 100%|##########| 3420/3420 [00:34<00:00, 99.67it/s] 


In [57]:
## Stitch all chunk PLYs into one mesh
# Use this script to merge chunk mesh pieces into one large mesh
#
# Parameters are:
# --input-dir tmp_dbg/meshes         (Adjust to dir where your meshes are)
# --output merged_segment1_debug.ply (Adjust to where you want to save the merged mesh and the name you want)
# --tol 1e-6                         (Tolerance for distance merging, tol of 1e-6 will more or less only merge exact overlaps, higher values will merge more freely, may reduce mesh complexity but reduce resolution)
!python glue_and_scale_mesh----final_v01.py \
    --input-dir tmp_dbg_12/meshes \
    --output merged_segment1_debug.ply \
    --tol 1e-2


Loading 1071 chunk meshes...
Concatenating meshes...
Merging vertices (tol=0.01)...
Removing duplicate faces...
Exporting merged mesh to merged_segment1_debug_8.ply ...
Done!


In [3]:
%%bash
# export the current env to a clean YAML (dropping the absolute prefix)
conda env export --no-builds \
  | grep -v '^prefix: ' \
  > environment.yml
echo "Wrote environment.yml in your notebook folder"


-bash: line 2: conda: command not found


Wrote environment.yml in your notebook folder
